<a href="https://colab.research.google.com/github/JabulaniMcineka/MyProjects/blob/main/Sports_Data_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import json
import boto3
from google.colab import userdata

# ---- GET RAW DATA FROM S3 ----
AWS_ACCESS_KEY = userdata.get('AWS_ACCESS_KEY')
AWS_SECRET_KEY = userdata.get('AWS_SECRET_KEY')
BUCKET_NAME_RAW = "sports-data-raw-7977-9545-4172"
BUCKET_NAME_TRANSFORMED = "sports-data-transformed-7977-9545-4172"

s3 = boto3.client(
    's3',
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    region_name='us-east-1'
)

# Get the file we just saved
obj = s3.get_object(
    Bucket=BUCKET_NAME_RAW,
    Key="raw/premier_league_20260316_124310.json"
)

data = json.loads(obj['Body'].read())
print(" Raw data loaded from S3!")

# ---- TRANSFORM DATA ----
events = data['events']

df = pd.DataFrame([{
    'event_id': e['idEvent'],
    'date': e['strTimestamp'],
    'home_team': e['strEvent'].split(' vs ')[0],
    'away_team': e['strEvent'].split(' vs ')[1],
    'league': e['strLeague'],
    'sport': e['strSport'],
    'season': e.get('strSeason', 'N/A')
} for e in events])

print(" Data transformed successfully!")
print(f"Total records: {len(df)}")
print(df.head())

# ---- SAVE TRANSFORMED DATA TO S3 ----
transformed_filename = "premier_league_transformed.json"

s3.put_object(
    Bucket=BUCKET_NAME_TRANSFORMED,
    Key=f"transformed/{transformed_filename}",
    Body=df.to_json(orient='records'),
    ContentType='application/json'
)

print(f" Transformed data saved to S3!")

AccessDenied: An error occurred (AccessDenied) when calling the GetObject operation: User: arn:aws:iam::797795454172:user/itsJabulaniadmin is not authorized to perform: s3:GetObject on resource: "arn:aws:s3:::sports-data-raw-7977-9545-4172/raw/premier_league_20260316_124310.json" with an explicit deny in an identity-based policy

In [ ]:
import requests
import json
import boto3
from datetime import datetime
from google.colab import userdata

# ---- SPORTS API ----
url = "https://www.thesportsdb.com/api/v1/json/3/eventspastleague.php"
params = {"id": "4328"}

response = requests.get(url, params=params)
data = response.json()
print("Data fetched successfully!")
print(f"Total events: {len(data['events'])}")

# ---- AWS CREDENTIALS FROM SECRETS ----
AWS_ACCESS_KEY = userdata.get('AWS_ACCESS_KEY')
AWS_SECRET_KEY = userdata.get('AWS_SECRET_KEY')
BUCKET_NAME = "sports-data-raw-7977-9545-4172"

# ---- SAVE TO S3 ----
s3 = boto3.client(
    's3',
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    region_name='us-east-1'
)

filename = f"premier_league_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"

s3.put_object(
    Bucket=BUCKET_NAME,
    Key=f"raw/{filename}",
    Body=json.dumps(data),
    ContentType='application/json'
)

print(f"Data saved to S3: raw/{filename}")

Data fetched successfully!
Total events: 15
Data saved to S3: raw/premier_league_20260316_144503.json


In [ ]:
import boto3
from google.colab import userdata

# ---- CREDENTIALS ----
AWS_ACCESS_KEY = userdata.get('AWS_ACCESS_KEY')
AWS_SECRET_KEY = userdata.get('AWS_SECRET_KEY')
BUCKET_NAME_TRANSFORMED = "sports-data-transformed-7977-9545-4172"

s3 = boto3.client(
    's3',
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    region_name='us-east-1'
)

# ---- SAVE TRANSFORMED DATA TO S3 ----
transformed_filename = "premier_league_transformed.json"

s3.put_object(
    Bucket=BUCKET_NAME_TRANSFORMED,
    Key=f"transformed/{transformed_filename}",
    Body=df.to_json(orient='records'),
    ContentType='application/json'
)

print(" Transformed data saved to S3!")

 Transformed data saved to S3!


In [ ]:
import boto3
import pandas as pd
import json
import sqlite3
from google.colab import userdata

try:
    AWS_ACCESS_KEY = userdata.get('AWS_ACCESS_KEY')
    AWS_SECRET_KEY = userdata.get('AWS_SECRET_KEY')

    s3 = boto3.client(
        "s3",
        aws_access_key_id=AWS_ACCESS_KEY,
        aws_secret_access_key=AWS_SECRET_KEY,
        region_name="us-east-1"
    )

    obj = s3.get_object(
        Bucket="sports-data-transformed-7977-9545-4172",
        Key="transformed/premier_league_transformed.json"
    )

    data = json.loads(obj["Body"].read().decode("utf-8"))

    df = pd.DataFrame(data)

    print("Records:", len(df))

    conn = sqlite3.connect("sports_data.db")

    df.to_sql("premier_league_events", conn, if_exists="replace", index=False)

    result = pd.read_sql_query(
        "SELECT * FROM premier_league_events LIMIT 5",
        conn
    )

    print(result)

    conn.close()

except Exception as e:
    print("Pipeline failed:", e)

Records: 15
  event_id                 date          home_team         away_team  \
0  2275077  2026-03-14T15:00:00        Exeter City      Cardiff City   
1  2275076  2026-03-14T15:00:00  Wycombe Wanderers        Luton Town   
2  2275075  2026-03-14T15:00:00     Wigan Athletic     Bradford City   
3  2275074  2026-03-14T15:00:00          Stevenage     AFC Wimbledon   
4  2275073  2026-03-14T12:30:00   Rotherham United  Bolton Wanderers   

             league   sport     season  
0  English League 1  Soccer  2025-2026  
1  English League 1  Soccer  2025-2026  
2  English League 1  Soccer  2025-2026  
3  English League 1  Soccer  2025-2026  
4  English League 1  Soccer  2025-2026  
